In [2]:
import mysql.connector

RDS_HOST = "ecommerce-recommendation-db.c9u0ucy2umao.ap-south-1.rds.amazonaws.com"
RDS_PORT = 3306

# Change this after you create the database
RDS_DATABASE = "ecommerce_analytics"

RDS_USER = "adminuser"

# Replace with the password you created for the RDS instance
RDS_PASSWORD = "aimecesar5"

connection = mysql.connector.connect(
    host=RDS_HOST,
    port=RDS_PORT,
    database=RDS_DATABASE,
    user=RDS_USER,
    password=RDS_PASSWORD,
    connection_timeout=15
)

print("Connected to Amazon RDS MySQL successfully.")

connection.close()

Connected to Amazon RDS MySQL successfully.


In [3]:
from pyspark.sql import SparkSession

MYSQL_JAR = (
    r"C:\Spark\jars\mysql-connector-j-9.x.x.jar"
)

spark = (
    SparkSession.builder
    .appName("EcommerceRDSExport")
    .master("local[2]")
    .config("spark.jars", MYSQL_JAR)
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

In [1]:
import os
import getpass

os.environ["RDS_PASSWORD"] = getpass.getpass(
    "Enter the Amazon RDS password: "
)

Enter the Amazon RDS password:  ········


In [2]:
RDS_HOST = "YOUR-RDS-ENDPOINT"
RDS_PORT = 3306
RDS_DATABASE = "ecommerce_analytics"
RDS_USER = "adminuser"
RDS_PASSWORD = os.environ["RDS_PASSWORD"]

JDBC_URL = (
    f"jdbc:mysql://{RDS_HOST}:{RDS_PORT}/{RDS_DATABASE}"
    "?useSSL=true"
    "&requireSSL=true"
    "&serverTimezone=UTC"
    "&rewriteBatchedStatements=true"
)

JDBC_PROPERTIES = {
    "user": RDS_USER,
    "password": RDS_PASSWORD,
    "driver": "com.mysql.cj.jdbc.Driver"
}

In [3]:
import getpass
import mysql.connector

RDS_HOST = (
    "ecommerce-recommendation-db."
    "c9u0ucy2umao.ap-south-1.rds.amazonaws.com"
)
RDS_PORT = 3306
RDS_USER = "adminuser"
RDS_PASSWORD = getpass.getpass("Enter RDS password: ")

connection = mysql.connector.connect(
    host=RDS_HOST,
    port=RDS_PORT,
    user=RDS_USER,
    password=RDS_PASSWORD,
    connection_timeout=20
)

cursor = connection.cursor()

cursor.execute(
    """
    CREATE DATABASE IF NOT EXISTS ecommerce_bigdata
    CHARACTER SET utf8mb4
    COLLATE utf8mb4_unicode_ci
    """
)

connection.commit()

print("Database ecommerce_bigdata is ready.")

cursor.close()
connection.close()

Enter RDS password:  ········


Database ecommerce_bigdata is ready.


In [8]:
ANALYTICS_ROOT = (
    "hdfs://localhost:9000/ecommerce/analytics"
)

MODEL_ROOT = (
    "hdfs://localhost:9000/ecommerce/models"
)

recommendations_hdfs = spark.read.parquet(
    f"{ANALYTICS_ROOT}/hybrid_recommendations"
)

evaluation_hdfs = spark.read.parquet(
    f"{ANALYTICS_ROOT}/recommendation_evaluation"
)

In [9]:
from pyspark.sql.functions import (
    col,
    current_timestamp,
    round as spark_round
)

recommendations_rds = (
    recommendations_hdfs
    .select(
        col("user_id").cast("string"),
        col("recommendation_rank")
        .cast("integer"),
        col("product_id").cast("string"),
        col("product_name").cast("string"),
        col("category_name").cast("string"),
        spark_round(
            col("als_score_normalized"),
            6
        ).cast("double").alias("als_score"),
        spark_round(
            col("content_score_normalized"),
            6
        ).cast("double").alias(
            "content_score"
        ),
        spark_round(
            col("hybrid_score"),
            6
        ).cast("double").alias(
            "hybrid_score"
        )
    )
    .withColumn(
        "generated_at",
        current_timestamp()
    )
)

In [10]:
from pyspark.sql.functions import (
    avg,
    lit
)

evaluation_summary_rds = (
    evaluation_hdfs
    .agg(
        avg("precision_at_10").alias(
            "mean_precision_at_10"
        ),
        avg("recall_at_10").alias(
            "mean_recall_at_10"
        )
    )
    .withColumn(
        "model_name",
        lit("hybrid_als_content")
    )
    .withColumn(
        "evaluated_at",
        current_timestamp()
    )
)

In [11]:
from pyspark.sql import SparkSession

MYSQL_JAR = (
    r"C:\Spark\jars\mysql-connector-j-9.4.0.jar"
)

spark = (
    SparkSession.builder
    .appName("HDFSToAmazonRDS")
    .master("local[2]")
    .config("spark.jars", MYSQL_JAR)
    .config(
        "spark.sql.shuffle.partitions",
        "8"
    )
    .getOrCreate()
)

In [12]:
RDS_DATABASE = "ecommerce_bigdata"

JDBC_URL = (
    f"jdbc:mysql://{RDS_HOST}:"
    f"{RDS_PORT}/{RDS_DATABASE}"
    "?useSSL=true"
    "&serverTimezone=UTC"
    "&rewriteBatchedStatements=true"
)

JDBC_OPTIONS = {
    "url": JDBC_URL,
    "user": RDS_USER,
    "password": RDS_PASSWORD,
    "driver": "com.mysql.cj.jdbc.Driver"
}

In [14]:
(
    recommendations_rds
    .coalesce(4)
    .write
    .format("jdbc")
    .option("url", JDBC_URL)
    .option(
        "dbtable",
        "recommendations"
    )
    .option("user", RDS_USER)
    .option(
        "password",
        RDS_PASSWORD
    )
    .option(
        "driver",
        "com.mysql.cj.jdbc.Driver"
    )
    .option("batchsize", "2000")
    .option(
        "isolationLevel",
        "READ_COMMITTED"
    )
    .mode("overwrite")
    .save()
)

print(
    "Recommendations saved to Amazon RDS."
)

Recommendations saved to Amazon RDS.


In [15]:
(
    evaluation_summary_rds
    .write
    .format("jdbc")
    .option("url", JDBC_URL)
    .option(
        "dbtable",
        "model_evaluation"
    )
    .option("user", RDS_USER)
    .option(
        "password",
        RDS_PASSWORD
    )
    .option(
        "driver",
        "com.mysql.cj.jdbc.Driver"
    )
    .mode("append")
    .save()
)

print(
    "Evaluation summary saved to Amazon RDS."
)

Evaluation summary saved to Amazon RDS.


In [16]:
recommendations_check = (
    spark.read
    .format("jdbc")
    .option("url", JDBC_URL)
    .option(
        "dbtable",
        "recommendations"
    )
    .option("user", RDS_USER)
    .option(
        "password",
        RDS_PASSWORD
    )
    .option(
        "driver",
        "com.mysql.cj.jdbc.Driver"
    )
    .load()
)

evaluation_check = (
    spark.read
    .format("jdbc")
    .option("url", JDBC_URL)
    .option(
        "dbtable",
        "model_evaluation"
    )
    .option("user", RDS_USER)
    .option(
        "password",
        RDS_PASSWORD
    )
    .option(
        "driver",
        "com.mysql.cj.jdbc.Driver"
    )
    .load()
)

print(
    "RDS recommendation rows:",
    recommendations_check.count()
)

print(
    "RDS evaluation rows:",
    evaluation_check.count()
)

RDS recommendation rows: 100000
RDS evaluation rows: 1


In [5]:
import pymysql

conn = pymysql.connect(
    host=RDS_HOST,
    port=RDS_PORT,
    user=RDS_USER,
    password=RDS_PASSWORD,
    database=RDS_DATABASE,
    charset="utf8mb4",
    cursorclass=pymysql.cursors.DictCursor,
    connect_timeout=20,
    autocommit=False,
    ssl_disabled=True
)'

print("Connected successfully to Amazon RDS!")

SyntaxError: unterminated string literal (detected at line 14) (2465006283.py, line 14)